# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JAN-tech404/FlyRank.internee/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

I choose **ranking / scoring**.

**Why ranking, not classification.** My lane (Lane 2 — Refresh / Content Opportunity
Scoring, picked in w01) answers an editor's question: *out of thousands of pages, which
ones should I look at FIRST this week?* That is a "which ones first?" question — the
textbook setup for a ranker. The output is a **continuous priority score per page**, not
a yes/no class label. A classifier that predicts "is this declining?" would tell the
editor *what to label*, not *what to do next week*.

**Why not clustering.** Clustering groups pages by shape (e.g. "thin stale low-CTR
cluster"). Useful for exploration, useless for the weekly action queue — the editor
still needs a per-page ordering inside whichever cluster matters.

**Why not pure signal analysis.** I will use signal analysis (Lane 6) to *audit* which
features carry weight, but the deliverable is a per-page score that drives a queue.
Scoring/ranking is the wrapper.

**What ranking buys us, concretely.** Given a fixed review budget (say 20 pages/week),
a ranker tries to put the most *actionable* pages in the top 20. The metric that
matches this budget constraint is **precision@K**, not accuracy.

In [ ]:
# Section 1 — sanity: the data is one-page-per-row
import pandas as pd

df = pd.read_csv(
    "/home/jan/Git cloned repos/FlyRank.internee/data/raw/content_refresh_anonymized.csv"
)

print(f"Rows: {len(df):,}")
print(f"Unique content_id: {df['content_id'].nunique():,}  (1 row = 1 content item)")
print(f"Unique clients: {df['client_id'].nunique()}")
print(f"Columns: {df.shape[1]}")

# Confirm trend_direction and trend_pct exist (the leakage trap we MUST avoid as features)
print(f"trend_direction values: {df['trend_direction'].value_counts().to_dict()}")
print(f"is_declining_label derived: {(df['trend_direction']=='down').mean()*100:.1f}% declining")


## 2. Target or proxy

**The target column:** `opportunity_score` (continuous, 0–1) — a per-page priority score
that the model learns to produce directly. For *validation* I use a separately defined
proxy positive set: pages that are **visible, under-performing on CTR, and stale** —
the kind of page a refresh would plausibly help. The proxy is *not* a training label;
it is the held-out yardstick.

**Why not `is_declining_label` (the obvious choice).** The data dictionary is explicit:
`is_declining_label = (trend_direction == "down")` and `trend_direction` is itself
derived from `trend_pct`. Training against it would make the model learn a function of
`trend_pct` — a column that is the label, not a cause. That's the textbook leakage trap,
and the same trap FlyRank's product flags fall into: the model would just *reproduce
the rule that already exists*, not surface anything new.

**Where the proxy positives come from (all observed, all from the row's own data):**

| Component | Column | Threshold | Why |
|---|---|---|---|
| Visible | `impressions_90d` | ≥ 100 | Page gets real search demand — refreshing it can pay off |
| Under-performing | `ctr` | < 0.5% | For a visible page, this CTR is the "room to grow" signal |
| Stale | `days_since_last_update` | > 90 | Page hasn't been touched in a quarter — refresh is plausible |
| Thin (optional) | `word_count` | < 2000 or blank | Suggests there is content to add, not just a re-check |

A page is a proxy positive if it hits all three of visible + under-performing + stale.
These are **observed measurements on the row**, not a pipeline rule.

**The honest word.** I will call this `opportunity_proxy` and treat it as a
*yardstick for ranking quality*, not as ground truth. The model's score is what an
editor sees; the proxy is what I check precision against.

In [ ]:
# Section 2 — build the proxy positive set the model will be ranked against
# All three components are observed on the row, not derived from trend_*
import pandas as pd

df = pd.read_csv(
    "/home/jan/Git cloned repos/FlyRank.internee/data/raw/content_refresh_anonymized.csv"
)

# visible + under-performing + stale  (all thresholds are on raw observed columns)
visible     = df['impressions_90d']       >= 100
under_ctr   = df['ctr']                   <  0.5
stale       = df['days_since_last_update'] >  90

df['opportunity_proxy'] = (visible & under_ctr & stale).astype(int)

# A second, leaner variant that also requires the page to have measurable position
# (avg_position > 0 means GSC has ranked it; the data dictionary flags 0 as "no data")
ranked = df['avg_position'] > 0
df['opportunity_proxy_ranked'] = (visible & under_ctr & stale & ranked).astype(int)

print(f"Total pages: {len(df):,}")
print(f"Proxy positives (visible & low-CTR & stale): {df['opportunity_proxy'].sum():,} "
      f"({df['opportunity_proxy'].mean()*100:.1f}%)")
print(f"Proxy positives (visible & low-CTR & stale & ranked): "
      f"{df['opportunity_proxy_ranked'].sum():,} "
      f"({df['opportunity_proxy_ranked'].mean()*100:.1f}%)")

# Sanity: are proxy positives spread across clients? (one client shouldn't dominate)
n_clients_with_pos = df.loc[df['opportunity_proxy']==1, 'client_id'].nunique()
print(f"Clients with at least one proxy-positive page: {n_clients_with_pos} / {df['client_id'].nunique()}")


## 3. Success metric

**Primary metric: `precision@K`** (specifically `precision@20`, matching a realistic
weekly review budget of 20 pages per editor).

`precision@20` = (number of proxy-positive pages in the top-20 by score) / 20.

**Why precision@K and not accuracy / AUC / F1:**
- *Accuracy* ignores the ranking — the editor only sees the top of the list, so what
  matters is whether the top contains real opportunities, not the overall error rate.
- *AUC* measures the model's ability to separate positives from negatives *across the
  whole ranking*; my decision is concentrated at the top, so the top matters more.
- *F1* needs a binary cut-off; a ranker is better evaluated at a budget (K) than at a
  threshold.

**Number that means "good":** a useful working target is **`precision@20 ≥ 0.50`**
(≥ 10 of the editor's 20 weekly slots land on a real refresh candidate), with the
hard floor that the score-based ranking must beat a hand-written "stale-and-visible"
rule on the same validation split. I'll report this on a **client-holdout** split
(train on N−1 clients, test on the held-out client) — a single random split would
let the model memorize per-client quirks.

**What I will NOT optimize for:** recall at the corpus level. Catching 100% of
opportunities would require ranking every page; the editor can't review every page,
so the question is the *top of the list*, not the tail.

In [ ]:
# Section 3 — measure the BASELINE so 'good' has a number
import pandas as pd
import numpy as np

df = pd.read_csv(
    "/home/jan/Git cloned repos/FlyRank.internee/data/raw/content_refresh_anonymized.csv"
)

# proxy positive (re-derive for the cell to be self-contained)
visible   = df['impressions_90d']       >= 100
under_ctr = df['ctr']                   <  0.5
stale     = df['days_since_last_update'] >  90
proxy     = (visible & under_ctr & stale).astype(int)

# Baseline rule: "stale & visible" — what a hand-written if-statement would produce
# (rank by impressions within the rule set; ties broken arbitrarily)
baseline_score = (
    (df['days_since_last_update'] > 90).astype(int) * 1_000_000
    + df['impressions_90d'].fillna(0)
)

# precision@K evaluator
def precision_at_k(scores, labels, k):
    order = np.argsort(-scores.values)[:k]
    return labels.iloc[order].mean()

K = 20
p_at_20_rule = precision_at_k(baseline_score, proxy, K)
p_at_20_random = proxy.mean()  # expected for a random ranking

print(f"Proxy-positive prevalence (random baseline): {p_at_20_random:.3f}")
print(f"precision@20 for the 'stale & visible' rule: {p_at_20_rule:.3f}")
print(f"Working target for the model: precision@20 >= 0.50 (>=10 of 20 real opportunities)")


## 4. The unit of analysis, as a real dataframe

**One row = one content item (a single page published for one client).**

Below is the actual slice of the starter dataset my lane cares about, with the
features (signals) a model would learn from and the proxy target it would be ranked
against. The columns are the **row's own measurements** — no joins, no aggregations.

**Why a one-page row.** The weekly review queue is a list of *pages* to look at. The
unit of analysis has to match the unit of action. Clustering by client or by topic
would group the action incorrectly (a single client is not "the" action — the action
is a page-level edit).

In [ ]:
# Section 4 — show the unit of analysis as a real dataframe
import pandas as pd
import numpy as np

df = pd.read_csv(
    "/home/jan/Git cloned repos/FlyRank.internee/data/raw/content_refresh_anonymized.csv"
)

# Build the proxy target (visible + low-CTR + stale) so we can see it next to features
df['opportunity_proxy'] = (
    (df['impressions_90d']       >= 100) &
    (df['ctr']                   <  0.5) &
    (df['days_since_last_update'] >  90)
).astype(int)

# A sketch of the feature set the model would use.
# EXCLUDED on purpose (the leakage trap):
#   - trend_direction, trend_pct        -> they ARE the label
#   - content_id, client_id             -> pseudonyms; for grouping/splitting only
#   - provider_used, model_used        -> generator metadata, not page quality
FEATURE_COLS = [
    # demand (visible means the page is reachable)
    'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d',
    # performance rates (×100 percentages; per data dictionary)
    'ctr', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct',
    # ranking (avg_position == 0 means "no data", not position zero)
    'avg_position',
    # freshness / depth
    'content_age_days', 'days_since_last_update',
    'word_count', 'char_count',
    # keyword context (lots of missingness along content_type — handle later)
    'search_volume', 'competition', 'cpc',
]

view_cols = ['content_id', 'client_id'] + FEATURE_COLS + ['opportunity_proxy']
unit_df = df[view_cols].head(8).copy()

# Show the *shape* of one page's row — feature values + the proxy target
print("One row = one content item.  Sample (8 rows, all columns present in the slice):")
display_cols = FEATURE_COLS + ['opportunity_proxy']
print(unit_df[display_cols].to_string(index=False))
print(f"\nUnit-of-analysis slice shape: {unit_df.shape}")
print(f"  -> {unit_df.shape[0]} pages shown x {len(display_cols)} columns (features + proxy target)")
print(f"  -> proxy=1 means: visible (>=100 imp) AND low CTR (<0.5%) AND stale (>90 days)")


## 5. Why ML beats a fixed rule here

A fixed rule (`IF stale AND visible AND low-CTR THEN review`) is the natural starting
point — and in fact it IS my baseline (Section 3). It works because the three signals
happen to be the strongest, most interpretable, and cheapest to compute. So why
bother with a model? Three reasons that come straight out of the data:

1. **Threshold drift.** "Stale" is one number, but the editor cares about the
   *interaction* of staleness with depth and traffic. A 1,200-word page that hasn't
   been touched in 100 days is very different from a 1,200-word page that hasn't
   been touched in 800 days. A rule with one threshold per axis has to enumerate
   those interactions by hand; a model learns them.
2. **Missingness is structured, not random.** Per the data dictionary, the
   keyword-context columns are missing along `content_type` lines, and `word_count`
   is missing for ~26% of rows. A blind `fillna(0)` would inject a content-type
   signal into the features (a hidden leakage-of-the-imputation kind). A model
   pipeline that uses `has_*` flags and per-content-type imputation behaves
   correctly where a hand-written rule silently misfires.
3. **Many signals at once, none dominant.** A quick look (see code) shows that
   *no single column* picks out the proxy positives cleanly — the best single
   feature lifts precision over the random baseline only modestly. The right
   answer is a weighted combination, and the weights aren't equal. A learned
   scorer finds the weights; a rule picks them by gut feel.

**Where a rule still wins.** When the editor needs an explanation in one line, a
rule is more honest than a 12-feature gradient-boosted score. That's why the
deliverable is a **scored ranking with reason codes**, not a black-box score:
the model proposes the queue, the rule language is what the editor reads.

**The one-paragraph frame** (the test from the framing skill):

> For an **editor**, deciding **which page to refresh first** under a fixed weekly
> budget, we will build a **per-page priority score** from one row of observed
> search/analytics signals, ranking **proxy-positive** pages higher, measured by
> **precision@20** on a client-holdout split. A wrong call costs **a wasted review
> slot that could have gone to a higher-value page**. A plain rule isn't enough
> because **the signals interact and missingness is structured**; a learned scorer
> finds the weighting, and reason codes keep the output honest.

In [ ]:
# Section 5 — show why one feature alone is not enough (the case for a model)
import pandas as pd
import numpy as np

df = pd.read_csv(
    "/home/jan/Git cloned repos/FlyRank.internee/data/raw/content_refresh_anonymized.csv"
)
proxy = (
    (df['impressions_90d']       >= 100) &
    (df['ctr']                   <  0.5) &
    (df['days_since_last_update'] >  90)
).astype(int)

def precision_at_k(scores, k):
    order = np.argsort(-scores.values)[:k]
    return proxy.iloc[order].mean()

K = 20
prevalence = proxy.mean()

# each of these is "rank by this one signal, take top 20"
single_feature_results = {
    "impressions_90d (more visible first)":
        precision_at_k(df['impressions_90d'].fillna(0), K),
    "ctr (lower CTR first)":
        precision_at_k(-df['ctr'].fillna(99), K),
    "days_since_last_update (older first)":
        precision_at_k(df['days_since_last_update'].fillna(0), K),
    "content_age_days (older first)":
        precision_at_k(df['content_age_days'].fillna(0), K),
    "word_count (thinner first, NaN -> big)":
        precision_at_k(-df['word_count'].fillna(999999), K),
    "search_volume (higher demand first)":
        precision_at_k(df['search_volume'].fillna(0), K),
    "avg_position (worse rank first)":
        precision_at_k(df['avg_position'].fillna(0), K),
}

print(f"Random baseline precision@20 (prevalence): {prevalence:.3f}\n")
print(f"{'Single-signal ranking':<42}  precision@20")
print("-" * 60)
for name, score in sorted(single_feature_results.items(), key=lambda kv: -kv[1]):
    lift = (score - prevalence) / max(prevalence, 1e-9) * 100
    print(f"{name:<42}  {score:.3f}   (lift {lift:+.0f}%)")

print("\nNo single signal reaches the 0.50 working target. The signal is real but")
print("messy — the right answer is a weighted combination, which is what a model learns.")


## 6. Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Run all in Colab)
- [x] No client names, URLs, or private queries anywhere — file paths in this repo
      are public, the dataset is the pseudonymized starter, and Hugging Face
      warehouse access is not used in this notebook
- [x] Claims use careful words: observed, measured, directional, decision-support —
      no "predicts Google", no causal language
- [x] Committed to my repo under `work/notebooks/`

**Honest scope note.** This notebook *frames* the ML task: it picks the task type
(ranking/scoring), names a measurable proxy target from observed row-level signals,
and sets the success metric (precision@20 on a client-holdout split). It does NOT
yet train the model — that is the work of w04 (baseline) and w05 (model), which
build on this frame. The frame is what locks the rest of the project to the right
question.